In [1]:
import torch
import torch.nn.functional as F

def vanilla_attention(Q, K, V):
    """
    Q: (batch, seq_len_q, d)
    K: (batch, seq_len_k, d)
    V: (batch, seq_len_k, d_v)
    """
    # 1. Compute raw scores (dot product)
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch, seq_len_q, seq_len_k)
    print("\nScores:", scores)
    # 2. Scale by sqrt(d) for stability
    d = K.size(-1)
    print("\nDimension d:", d)
    scores = scores / torch.sqrt(torch.tensor(d, dtype=torch.float32))
    print("\nScaled Scores:", scores)
    # 3. Softmax across keys
    attn_weights = F.softmax(scores, dim=-1)       # (batch, seq_len_q, seq_len_k)
    print("\nAttention Weights:", attn_weights)
    # 4. Weighted sum of values
    output = torch.matmul(attn_weights, V)         # (batch, seq_len_q, d_v)

    return output, attn_weights


In [2]:
# Example: batch=1, seq_len=3, feature_dim=4
batch, seq_len, d = 1, 3, 4
Q = torch.randn(batch, seq_len, d)
K = torch.randn(batch, seq_len, d)
V = torch.randn(batch, seq_len, d)
print("Q:", Q)
print("\nK:", K)
print("\nV:", V)
out, attn = vanilla_attention(Q, K, V)

print("\nOutput:", out.shape)         # (1, 3, 4)
print("\nOutput Values:", out)
print("\nAttention Weights:", attn.shape)  # (1, 3, 3)
print("\nAttention Weights Values:", attn)


Q: tensor([[[-2.9084,  1.2275,  1.7534,  0.3976],
         [ 1.3671,  0.0282,  0.9339,  1.0425],
         [-1.2431,  0.1244, -0.1354,  0.5597]]])

K: tensor([[[ 0.5471,  0.1927,  0.3269,  0.0564],
         [-0.3113,  0.2764, -0.7132, -0.8697],
         [ 0.6172, -1.1582, -1.1973, -1.0665]]])

V: tensor([[[-1.1585e+00,  7.6512e-02, -1.9810e-03, -7.5809e-01],
         [-7.3180e-01, -5.9355e-01, -2.1573e+00, -1.1168e-01],
         [ 8.7420e-01,  8.9772e-01,  5.4942e-01,  5.6418e-01]]])

Scores: tensor([[[-0.7589, -0.3517, -5.7402],
         [ 1.1175, -1.9905, -1.4189],
         [-0.6688,  0.0311, -1.3461]]])

Dimension d: 4

Scaled Scores: tensor([[[-0.3795, -0.1758, -2.8701],
         [ 0.5587, -0.9953, -0.7094],
         [-0.3344,  0.0156, -0.6731]]])

Attention Weights: tensor([[[0.4331, 0.5310, 0.0359],
         [0.6699, 0.1416, 0.1885],
         [0.3193, 0.4531, 0.2276]]])

Output: torch.Size([1, 3, 4])

Output Values: tensor([[[-0.8590, -0.2498, -1.1266, -0.3674],
         [-0.7150,

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ---- Positional Encoding ----
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

# ---- Multi-Head Attention ----
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):
        batch_size = Q.size(0)

        # 1. Linear projections
        Q = self.W_q(Q)  # (batch, seq_len, d_model)
        K = self.W_k(K)
        V = self.W_v(V)

        # 2. Split into heads
        def split_heads(x):
            return x.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)  # (batch, heads, seq_len, d_k)

        # 3. Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (batch, heads, seq_q, seq_k)
        attn_weights = F.softmax(scores, dim=-1)
        out = torch.matmul(attn_weights, V)  # (batch, heads, seq_len, d_k)

        # 4. Concatenate heads
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # 5. Final linear
        return self.W_o(out), attn_weights


In [11]:
batch, seq_len, d_model, num_heads = 2, 5, 16, 4
x = torch.randn(batch, seq_len, d_model)

# Add positional encoding
pos_enc = PositionalEncoding(d_model)
x_pos = pos_enc(x)

# Multi-head attention
mha = MultiHeadAttention(d_model, num_heads)
out, attn = mha(x_pos, x_pos, x_pos)

print("Input shape:", x.shape)      # (2, 5, 16)
print("Output shape:", out.shape)   # (2, 5, 16)
print("Attention weights:", attn.shape)  # (2, 4, 5, 5)


Input shape: torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])
Attention weights: torch.Size([2, 4, 5, 5])


**why do the gradients of softmax vanish when the inputs are very large in magnitude, similar to sigmoid/tanh?**

---

## 🔹 1. Reminder: Softmax Function

For input vector $z \in \mathbb{R}^n$, the softmax is:

$$
\sigma(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

---

## 🔹 2. Gradient of Softmax

The Jacobian of softmax is:

$$
\frac{\partial \sigma(z_i)}{\partial z_j} = \sigma(z_i)(\delta_{ij} - \sigma(z_j))
$$

So:

* If $i = j$:

$$
\frac{\partial \sigma(z_i)}{\partial z_i} = \sigma(z_i)(1 - \sigma(z_i))
$$

* If $i \neq j$:

$$
\frac{\partial \sigma(z_i)}{\partial z_j} = -\sigma(z_i)\sigma(z_j)
$$

---

## 🔹 3. Behavior for Large Inputs

Imagine one input $z_k$ is **much larger** than all others:

* $\sigma(z_k) \approx 1$
* $\sigma(z_j) \approx 0$ for $j \neq k$

Now check the gradients:

* For the "winning" entry ($k$):

$$
\frac{\partial \sigma(z_k)}{\partial z_k} \approx 1 \cdot (1 - 1) = 0
$$

* For all others ($j \neq k$):

$$
\frac{\partial \sigma(z_i)}{\partial z_j} \approx 0 \cdot (something) = 0
$$

So **all gradients vanish** → the Jacobian is nearly zero.

---

## 🔹 4. Analogy with Sigmoid/Tanh

* **Sigmoid**:
  $\sigma(x) = \frac{1}{1+e^{-x}}$ → derivative $\sigma(x)(1-\sigma(x))$.
  For large $|x|$, sigmoid ≈ 0 or 1 → derivative ≈ 0.
* **Tanh**:
  Derivative = $1 - \tanh^2(x)$.
  For large $|x|$, tanh ≈ ±1 → derivative ≈ 0.
* **Softmax**:
  When one input dominates, it collapses to a “hard” one-hot vector → derivatives ≈ 0.

So in all three cases: **high-magnitude inputs saturate the function, killing gradients**.

---

## 🔹 5. Why This Matters

* Training becomes **slow or stuck** when gradients vanish.
* In deep nets: backpropagated gradients shrink layer by layer → **vanishing gradient problem**.
* In attention: if logits (scores) are very large, softmax becomes nearly one-hot → gradients vanish → poor learning.

---

## 🔹 6. Practical Fixes

* **Temperature scaling**:
  $\sigma(z/T)$ with $T > 1$ makes softmax smoother, avoiding extreme saturation.
* **Normalization (LayerNorm / BatchNorm)**: keeps logits in a reasonable range.
* **Gradient clipping**: prevents exploding logits that push softmax into saturation.

---

✅ **Summary:**
Softmax gradients vanish for large input magnitudes because the output distribution becomes nearly one-hot. This is mathematically the same saturation effect seen in sigmoid and tanh, where extreme inputs flatten the function and kill gradients.

---

Would you like me to also **visualize with a small plot** (softmax output + gradient vs input magnitude) so you can *see* this saturation effect?


In [5]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V):
    """
    Q: (batch, heads, seq_len_q, d_k)
    K: (batch, heads, seq_len_k, d_k)
    V: (batch, heads, seq_len_k, d_v)
    """
    # 1. Compute similarity scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))  
    # shape: (batch, heads, seq_len_q, seq_len_k)

    # 2. Normalize with softmax
    attn_weights = F.softmax(scores, dim=-1)

    # 3. Weighted sum of values
    output = torch.matmul(attn_weights, V)  
    # shape: (batch, heads, seq_len_q, d_v)

    return output, attn_weights


In [6]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):
        batch_size = Q.size(0)

        # 1. Project inputs
        Q = self.W_q(Q)  # (batch, seq_len, d_model)
        K = self.W_k(K)
        V = self.W_v(V)

        # 2. Split into heads
        def split_heads(x):
            return x.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        print(f"\nQ is {Q.shape}, K is {K.shape}, V is {V.shape}")
        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)
        # shapes: (batch, heads, seq_len, d_k)
        print(f"\nQ is {Q.shape}, K is {K.shape}, V is {V.shape}")
        # 3. Apply scaled dot-product attention
        out, attn_weights = scaled_dot_product_attention(Q, K, V)
        print(f"\nOutput is {out.shape}, Attention Weights are {attn_weights.shape}")
        # 4. Concatenate heads
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        print(f"\nConcatenated Output is {out.shape}")
        # 5. Final linear projection
        return self.W_o(out), attn_weights


In [7]:
batch, seq_len, d_model, num_heads = 2, 5, 16, 4
x = torch.randn(batch, seq_len, d_model)
mha = MultiHeadAttention(d_model, num_heads)
out, attn = mha(x, x, x)

print("Input shape:", x.shape)        # (2, 5, 16)
print("Output shape:", out.shape)     # (2, 5, 16)
print("Attention shape:", attn.shape) # (2, 4, 5, 5)



Q is torch.Size([2, 5, 16]), K is torch.Size([2, 5, 16]), V is torch.Size([2, 5, 16])

Q is torch.Size([2, 4, 5, 4]), K is torch.Size([2, 4, 5, 4]), V is torch.Size([2, 4, 5, 4])

Output is torch.Size([2, 4, 5, 4]), Attention Weights are torch.Size([2, 4, 5, 5])

Concatenated Output is torch.Size([2, 5, 16])
Input shape: torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])
Attention shape: torch.Size([2, 4, 5, 5])


In [8]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        """
        features: size of the embedding dimension (d_model)
        eps: small value to avoid division by zero
        """
        super(LayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(features))  # learnable scale
        self.beta = nn.Parameter(torch.zeros(features))  # learnable shift
        self.eps = eps

    def forward(self, x):
        # x: (batch, seq_len, features)
        
        # 1. Compute mean and variance across features (last dim)
        mean = x.mean(dim=-1, keepdim=True)          # (batch, seq_len, 1)
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # (batch, seq_len, 1)

        # 2. Normalize
        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        # 3. Apply learnable scale (gamma) and shift (beta)
        out = self.gamma * x_norm + self.beta
        return out


In [9]:
# Example: batch=2, seq_len=3, embedding_dim=4
x = torch.tensor([
    [[1.0, 2.0, 3.0, 4.0],
     [2.0, 3.0, 4.0, 5.0],
     [3.0, 4.0, 5.0, 6.0]],

    [[10.0, 20.0, 30.0, 40.0],
     [5.0, 5.0, 5.0, 5.0],
     [7.0, 8.0, 9.0, 10.0]]
])

layer_norm = LayerNorm(features=4)

out = layer_norm(x)
print("Input:\n", x)
print("\nOutput after LayerNorm:\n", out)


Input:
 tensor([[[ 1.,  2.,  3.,  4.],
         [ 2.,  3.,  4.,  5.],
         [ 3.,  4.,  5.,  6.]],

        [[10., 20., 30., 40.],
         [ 5.,  5.,  5.,  5.],
         [ 7.,  8.,  9., 10.]]])

Output after LayerNorm:
 tensor([[[-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416]],

        [[-1.3416, -0.4472,  0.4472,  1.3416],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [-1.3416, -0.4472,  0.4472,  1.3416]]], grad_fn=<AddBackward0>)


In [10]:
import torch
import math
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        # Create a matrix of shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             -(math.log(10000.0) / d_model))

        # Apply sine to even indices, cosine to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)   # even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)   # odd dimensions

        pe = pe.unsqueeze(0)  # shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]


In [11]:
class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super(LearnedPositionalEmbedding, self).__init__()
        self.pos_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.size()
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)  # (1, seq_len)
        pos_emb = self.pos_embedding(positions)  # (1, seq_len, d_model)
        return x + pos_emb


In [12]:
batch, seq_len, d_model = 2, 10, 16
x = torch.randn(batch, seq_len, d_model)

# Sinusoidal
pos_enc = PositionalEncoding(d_model)
x_with_pe = pos_enc(x)
print("With Sinusoidal Positional Encoding:", x_with_pe.shape)

# Learned
learned_pe = LearnedPositionalEmbedding(max_len=100, d_model=d_model)
x_with_lpe = learned_pe(x)
print("With Learned Positional Embedding:", x_with_lpe.shape)


With Sinusoidal Positional Encoding: torch.Size([2, 10, 16])
With Learned Positional Embedding: torch.Size([2, 10, 16])
